# F1 pole position predictor

## Import


In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

qualifying = pd.read_csv("F1(1950 - 2024) data\\raw\\qualifying.csv" , na_values=['\\N'])
races = pd.read_csv("F1(1950 - 2024) data\\raw\\races.csv", na_values=['\\N'])
drivers = pd.read_csv("F1(1950 - 2024) data\\raw\\drivers.csv", na_values=['\\N'])
constructors = pd.read_csv("F1(1950 - 2024) data\\raw\\constructors.csv", na_values=['\\N'])

print('Data loaded successfully')

Data loaded successfully


## Checking data

In [4]:
print(qualifying.shape)
qualifying.head()

(10494, 9)


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


## check null

In [5]:
qualifying.isnull().sum()

qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                156
q2               4647
q3               6865
dtype: int64

In [6]:
races.sample(5)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
901,904,2014,5,4,Spanish Grand Prix,2014-05-11,12:00:00,http://en.wikipedia.org/wiki/2014_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1038,1051,2021,20,78,Qatar Grand Prix,2021-11-21,14:00:00,http://en.wikipedia.org/wiki/2021_Qatar_Grand_...,2021-11-19,NaN,2021-11-19,NaN,2021-11-20,NaN,2021-11-20,NaN,NaN,NaN
1009,1022,2019,13,13,Belgian Grand Prix,2019-09-01,13:10:00,http://en.wikipedia.org/wiki/2019_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
563,564,1976,5,40,Belgian Grand Prix,1976-05-16,NaN,http://en.wikipedia.org/wiki/1976_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
687,688,1967,10,46,United States Grand Prix,1967-10-01,NaN,http://en.wikipedia.org/wiki/1967_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data manipulation

In [7]:
df = pd.merge(qualifying, races, how='left', on='raceId')
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
8128,8152,1019,815,211,11,15,1:26.649,1:26.928,NaN,2019,10,9,British Grand Prix,2019-07-14,13:10:00,http://en.wikipedia.org/wiki/2019_British_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2918,2920,270,79,25,3,13,1:24.738,NaN,NaN,1994,14,26,European Grand Prix,1994-10-16,NaN,http://en.wikipedia.org/wiki/1994_European_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2106,2107,227,44,27,9,15,1:21.509,NaN,NaN,1996,4,20,European Grand Prix,1996-04-28,NaN,http://en.wikipedia.org/wiki/1996_European_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
428,429,38,11,8,22,17,1:33.984,NaN,NaN,2007,3,3,Bahrain Grand Prix,2007-04-15,11:30:00,http://en.wikipedia.org/wiki/2007_Bahrain_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3481,3483,343,67,5,16,14,1:28.534,1:28.273,NaN,2010,7,5,Turkish Grand Prix,2010-05-30,11:00:00,http://en.wikipedia.org/wiki/2010_Turkish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop pre 2005

In [8]:
df = df[df['year'] >= 2005]

## Convert Str to seconds

In [9]:
df['q1'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [10]:
df['q1'] = df['q1'].str.split(':').str[0].astype(float) * 60 + df['q1'].str.split(':').str[1].astype(float)

df['q2'] = df['q2'].str.split(':').str[0].astype(float) * 60 + df['q2'].str.split(':').str[1].astype(float)

df['q3'] = df['q3'].str.split(':').str[0].astype(float) * 60 + df['q3'].str.split(':').str[1].astype(float)


In [11]:
df['q1'].head(10)

0    86.572
1    86.103
2    85.664
3    85.994
4    85.960
5    86.427
6    86.295
7    86.381
8    86.919
9    86.702
Name: q1, dtype: float64

## Filling nulls with per year medians

In [12]:
df['q1'] = df.groupby('year')['q1'].transform(lambda x: x.fillna(x.median()))
df['q2'] = df.groupby('year')['q2'].transform(lambda x: x.fillna(x.median()))
df['q3'] = df.groupby('year')['q3'].transform(lambda x: x.fillna(x.median()))


In [17]:
df[['q1', 'q2', 'q3']].isnull().sum()

q1    0
q2    0
q3    0
dtype: int64

## Filling q3 nulls with 0

In [13]:
df['q3'] = df['q3'].fillna(0)

## create reached_q2&q3 column

In [14]:
df['reached_q2'] = (df['q2'].notnull()).astype(int)
df['reached_q3'] = (df['q3'].notnull()).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3
362,363,35,22,11,17,15,72.548,73.139,86.7915,2008,18,18,Brazilian Grand Prix,2008-11-02,17:00:00,http://en.wikipedia.org/wiki/2008_Brazilian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
3473,3475,343,3,131,4,6,87.649,87.141,86.9520,2010,7,5,Turkish Grand Prix,2010-05-30,11:00:00,http://en.wikipedia.org/wiki/2010_Turkish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
8681,8722,1047,840,211,18,8,96.502,96.143,96.0460,2020,17,24,Abu Dhabi Grand Prix,2020-12-13,13:10:00,http://en.wikipedia.org/wiki/2020_Abu_Dhabi_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
8703,8744,1052,840,117,18,10,91.261,90.624,90.6010,2021,1,3,Bahrain Grand Prix,2021-03-28,15:00:00,http://en.wikipedia.org/wiki/2021_Bahrain_Gran...,2021-03-26,NaN,2021-03-26,NaN,2021-03-27,NaN,2021-03-27,NaN,NaN,NaN,1,1
10059,10117,1123,844,6,16,5,76.984,76.304,76.4350,2024,3,1,Australian Grand Prix,2024-03-24,04:00:00,https://en.wikipedia.org/wiki/2024_Australian_...,2024-03-22,01:30:00,2024-03-22,05:00:00,2024-03-23,01:30:00,2024-03-23,05:00:00,NaN,NaN,1,1


In [21]:
print(df[df['q3'].isnull()]['reached_q3'].value_counts())

Series([], Name: count, dtype: int64)


## create got_pole column

In [15]:
df['got_pole'] = (df['position'] == 1).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
5624,5627,888,807,15,11,10,91.1320,90.231,90.9080,2013,9,20,German Grand Prix,2013-07-07,12:00:00,http://en.wikipedia.org/wiki/2013_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
279,280,31,21,10,21,12,96.2800,96.698,86.7915,2008,14,14,Italian Grand Prix,2008-09-14,12:00:00,http://en.wikipedia.org/wiki/2008_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
7267,7290,976,826,5,26,11,102.9270,103.186,89.4800,2017,8,73,Azerbaijan Grand Prix,2017-06-25,13:00:00,http://en.wikipedia.org/wiki/2017_Azerbaijan_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
9053,9094,1070,853,210,9,19,79.3030,81.086,81.0545,2021,18,32,Mexico City Grand Prix,2021-11-07,19:00:00,http://en.wikipedia.org/wiki/2021_Mexican_Gran...,2021-11-05,NaN,2021-11-05,NaN,2021-11-06,NaN,2021-11-06,NaN,NaN,NaN,1,1,0
9214,9272,1077,848,3,23,20,87.4545,85.225,88.0995,2022,4,21,Emilia Romagna Grand Prix,2022-04-24,13:00:00,http://en.wikipedia.org/wiki/2022_Emilia_Romag...,2022-04-22,11:30:00,2022-04-23,10:30:00,NaN,NaN,2022-04-22,15:00:00,2022-04-23,14:30:00,1,1,0


In [16]:
df['got_pole'].value_counts()

got_pole
0    7872
1     394
Name: count, dtype: int64

In [18]:
df.groupby('year')['q2'].median().head(20)

year
2005    89.1590
2006    81.4840
2007    83.1205
2008    83.4555
2009    91.2220
         ...   
2020    79.9620
2021    81.0860
2022    85.2250
2023    84.4405
2024    83.8770
Name: q2, Length: 20, dtype: float64

In [22]:
for i, col in enumerate(df.columns):
    print(i, col)

pd.set_option('display.max_columns', None)
df.sample(20)

0 qualifyId
1 raceId
2 driverId
3 constructorId
4 number
5 position
6 q1
7 q2
8 q3
9 year
10 round
11 circuitId
12 name
13 date
14 time
15 url
16 fp1_date
17 fp1_time
18 fp2_date
19 fp2_time
20 fp3_date
21 fp3_time
22 quali_date
23 quali_time
24 sprint_date
25 sprint_time
26 reached_q2
27 reached_q3
28 got_pole


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
7822,7846,1004,844,15,16,7,93.924,93.4880,93.4190,2018,16,71,Russian Grand Prix,2018-09-30,11:10:00,http://en.wikipedia.org/wiki/2018_Russian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
3019,3021,2,13,6,3,16,95.642,91.2220,92.3950,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
10453,10511,1142,840,117,18,20,94.484,83.8770,83.2340,2024,22,80,Las Vegas Grand Prix,2024-11-23,06:00:00,https://en.wikipedia.org/wiki/2024_Las_Vegas_G...,2024-11-21,02:30:00,2024-11-21,06:00:00,2024-11-22,02:30:00,2024-11-22,06:00:00,NaN,NaN,1,1,0
10034,10092,1121,842,214,10,20,90.948,83.8770,83.2340,2024,1,3,Bahrain Grand Prix,2024-03-02,15:00:00,https://en.wikipedia.org/wiki/2024_Bahrain_Gra...,2024-02-29,11:30:00,2024-02-29,15:00:00,2024-03-01,12:30:00,2024-03-01,16:00:00,NaN,NaN,1,1,0
4693,4696,847,37,15,17,17,76.229,75.5870,90.3075,2011,7,7,Canadian Grand Prix,2011-06-12,17:00:00,http://en.wikipedia.org/wiki/2011_Canadian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519,520,42,26,5,19,20,73.712,83.1205,83.4460,2007,7,19,United States Grand Prix,2007-06-17,17:00:00,http://en.wikipedia.org/wiki/2007_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
6789,6812,953,836,209,94,20,77.452,90.8225,89.8090,2016,6,6,Monaco Grand Prix,2016-05-29,12:00:00,http://en.wikipedia.org/wiki/2016_Monaco_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
333,334,34,17,9,10,6,96.238,95.6860,97.0830,2008,17,17,Chinese Grand Prix,2008-10-19,07:00:00,http://en.wikipedia.org/wiki/2008_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8768,8809,1055,847,3,63,15,78.445,79.1540,81.0545,2021,4,4,Spanish Grand Prix,2021-05-09,13:00:00,http://en.wikipedia.org/wiki/2021_Spanish_Gran...,2021-05-07,NaN,2021-05-07,NaN,2021-05-08,NaN,2021-05-08,NaN,NaN,NaN,1,1,0


In [23]:
# Drivers who have a q3 time but reached_q3 is 0
suspicious = df[(df['q3'] > 0) & (df['reached_q3'] == 0)]
print(suspicious.shape)
suspicious[['driverId', 'raceId', 'position', 'q1', 'q2', 'q3', 'reached_q3']].head(10)

(0, 29)


,driverId,raceId,position,q1,q2,q3,reached_q3
